# Data Engineering

> 15 trillion tokens -- the training data volume for LLaMA 3. This number sounds enormous, but where does it come from? It certainly doesn't fall from the sky.
>
> This section starts from raw HTML on the internet and walks through the complete data cleaning pipeline: text extraction, quality filtering, deduplication, and data mixing -- why each step is needed and how it works.

The starting point of data engineering is Common Crawl -- a non-profit project that crawls the entire internet every month, producing roughly 500TB of raw data. This data is in HTML format, mixed with navigation bars, advertisements, JavaScript code, and comment sections.

Feeding this directly to a model is equivalent to feeding it garbage. We need a systematic approach to turn raw HTML into clean, trainable text. The entire process consists of five steps: text extraction -> quality filtering -> deduplication -> data mixing -> tokenize. The filtering criteria at each step directly affect the quality of the final model, so each step requires clear judgment rules.

This section walks through these five steps in order, using real data fragments at each step to show the changes before and after filtering.

## 0. Data Pipeline Overview

Let's look at the big picture first, then tackle each step:

```
  Common Crawl (raw HTML, ~500TB/month)
       |
       v
  +-------------+
  | 1. Text Extraction  |  HTML -> plain text, remove nav/ads/scripts
  +------+------+
         v
  +-------------+
  | 2. Language Filtering  |  Keep only target languages (e.g. English, Chinese)
  +------+------+
         v
  +-------------+
  | 3. Quality Filtering  |  Remove ads/nav/gibberish/too short/too long
  +------+------+
         v
  +-------------+
  | 4. Deduplication  |  Exact dedup + MinHash approximate dedup
  +------+------+
         v
  +-------------+
  | 5. Data Mixing  |  Common Crawl + Wiki + Books + Code
  +------+------+
         v
    Clean training data
```

## 1. Text Extraction: Pulling Content from HTML

#### 1.1 What's Inside Common Crawl?

Common Crawl crawls billions of web pages every month, stored in WARC (Web ARChive) format. Each WARC file contains a complete HTML page -- including the main content, but also a lot of content unrelated to it:

- Navigation bars: "Home | About | Contact Us"
- Ad popups: "Limited time offer! Click to buy!"
- JavaScript code: "function trackUser(){...}"
- CSS styles: ".sidebar { float: right; }"
- Footers: "Copyright 2024. All rights reserved."
- Comment sections: "First! Great article!"

A normal article is wrapped in all of this stuff. The task of text extraction is to identify and keep the main content while discarding everything else. This process is not fully automated -- modern tools (like trafilatura, resiliparse) combine HTML tag analysis with machine learning models to determine which content is the main body.

#### 1.2 Using a Simulated HTML to Demonstrate the Extraction Process

In [ ]:
# === Simulation: A typical Common Crawl HTML page ===
raw_html = """
<!DOCTYPE html>
<html>
<head>
  <title>What is Machine Learning? - AI Blog</title>
  <meta name="description" content="Learn ML basics">
  <script src="tracking.js"></script>
  <style>.ad { color: red; }</style>
</head>
<body>
  <nav>
    <a href="/">Home</a> |
    <a href="/about">About</a> |
    <a href="/contact">Contact</a>
  </nav>
  
  <div class="sidebar">
    <div class="ad">
      <h3>Sponsored</h3>
      <p>Buy the best AI course! 50% off today only!</p>
      <button>Click Here!</button>
    </div>
    <script type="text/javascript">
      var _gaq = _gaq || [];
      _gaq.push(['_setAccount', 'UA-12345-6']);
      _gaq.push(['_trackPageview']);
    </script>
  </div>
  
  <article>
    <h1>What is Machine Learning?</h1>
    <p>Machine learning is a subset of artificial intelligence
    that enables systems to learn and improve from experience
    without being explicitly programmed.</p>
    
    <p>The process of learning begins with observations or data,
    such as examples, direct experience, or instruction.</p>
    
    <p>Machine learning algorithms build a mathematical model
    based on sample data, known as "training data".</p>
  </article>
  
  <div class="comments">
    <p>User123: Great article!</p>
    <p>Bot456: Buy followers at cheap-followers.com!</p>
  </div>
  
  <footer>
    <p>Copyright 2024 AI Blog. All rights reserved.</p>
    <p>Terms of Service | Privacy Policy | Cookie Settings</p>
  </footer>
</body>
</html>
"""

print("=== Raw HTML ===")
print(raw_html[:500])
print("...")
print()
print("^ Mixed in: navigation bar, ads, JS tracking code, comment spam, footer")

In [ ]:
import re
# === Manual simulation of the text extraction process ===
print("=== Text Extraction: HTML -> Plain Text ===")
print()

# Step 1: Remove <script> and <style> tags and their contents
def remove_scripts_styles(html):
    """Remove script and style tags (including contents)"""
    html = re.sub(r'<script[^>]*>.*?</script>', '', html, flags=re.DOTALL | re.IGNORECASE)
    html = re.sub(r'<style[^>]*>.*?</style>', '', html, flags=re.DOTALL | re.IGNORECASE)
    return html

# Step 2: Remove all HTML tags
def strip_tags(html):
    """Remove all <...> tags"""
    return re.sub(r'<[^>]+>', '', html)

# Step 3: Clean up whitespace
def clean_whitespace(text):
    """Merge multiple blank lines, trim leading/trailing whitespace"""
    text = re.sub(r'[ \t]+', ' ', text)  # Merge consecutive spaces/tabs
    text = re.sub(r'\n\s*\n\s*\n+', '\n\n', text)  # Multiple blank lines -> one blank line
    return text.strip()

# Step by step
print("Step 1 -- Remove script/style tags:")
no_scripts = remove_scripts_styles(raw_html)
print(f"  Original: {len(raw_html)} chars -> After removal: {len(no_scripts)} chars")
print()

print("Step 2 -- Remove all HTML tags:")
plain_text = strip_tags(no_scripts)
print(f"  After tag removal: {len(plain_text)} chars")
print()

print("Step 3 -- Clean up whitespace:")
clean_text = clean_whitespace(plain_text)
print(f"  After whitespace cleanup: {len(clean_text)} chars")
print()

print("=" * 60)
print("Extraction result:")
print("=" * 60)
print(clean_text)
print("=" * 60)
print()
print("Note: The navigation bar (Home|About|Contact) is still there, ads are still there, comments are still there")
print("      -> These need to be removed in the subsequent 'quality filtering' step")

## 2. Quality Filtering

After text extraction, most of the content is still low quality. Think about what's in Common Crawl: auto-generated SEO pages, gibberish, pages with only navigation bars and no content, ad copy repeated thousands of times... If this content is used directly for training, the model would learn things worse than learning nothing at all.

Therefore, quality filtering is the most critical step in the entire pipeline: it's better to lose some good data than to let massive amounts of garbage into the training set. Common methods fall into two levels:

#### 2.1 Heuristic Rules (Quickly Filter Out Obvious Junk)

These rules don't require a model -- they directly check the statistical features of the text. Each rule targets a specific type of common junk:

In [ ]:
np.random.seed(42)
import numpy as np
# === Quality filtering rules: step-by-step demonstration ===
print("=== Quality Filtering Rules ===")
print()

# Prepare several 'pending review' texts
samples = [
    {"name": "Good article", "text": "Machine learning is a subset of artificial intelligence that enables systems to learn and improve from experience without being explicitly programmed. The field has grown rapidly since the 2010s, driven by advances in deep learning and the availability of large datasets."},
    {"name": "Ad", "text": "BUY NOW!!! Click here!!! Limited time offer!!! Subscribe today and get 50% OFF!!! Don't miss this opportunity!!!"},
    {"name": "TOC page", "text": "Chapter 1. Introduction. Chapter 2. Methods. Chapter 3. Results. Chapter 4. Discussion. Chapter 5. Conclusion. Appendix A. Appendix B. References."},
    {"name": "Too short", "text": "Hello world."},
    {"name": "Random chars", "text": "asdfjkl; qwerty zxcvbnm @#$%@# 123456789 !!!!!!!"},
    {"name": "Repeat template", "text": "This is a blog post.\n" * 30 + "unique content here"},
]

def quality_check(text):
    """Return (keep_or_not, discard_reason)"""
    
    # Rule 1: Length filtering
    words = text.split()
    if len(words) < 5:
        return False, f"Too short ({len(words)} words < 5)"
    if len(text) > 5000:
        return False, f"Too long ({len(text)} chars > 5000)"
    
    # Rule 2: Average word length -- normal English word length is 3-10, gibberish words are very long
    avg_word_len = np.mean([len(w) for w in words])
    if avg_word_len > 12:
        return False, f"Abnormal avg word length ({avg_word_len:.1f} > 12)"
    
    # Rule 3: Special character ratio -- normal articles don't have punctuation exceeding 15%
    special_count = sum(1 for c in text if not c.isalnum() and not c.isspace())
    special_ratio = special_count / max(len(text), 1)
    if special_ratio > 0.25:
        return False, f"Too many special chars ({special_ratio:.1%} > 25%)"
    
    # Rule 4: Line repetition rate -- template pages have many repeated lines
    lines = [l.strip() for l in text.split('\n') if l.strip()]
    if len(lines) > 3:
        unique_ratio = len(set(lines)) / len(lines)
        if unique_ratio < 0.4:
            return False, f"Line repetition too high ({unique_ratio:.1%} < 40%)"
    
    # Rule 5: Uppercase letter ratio -- all CAPS LOCK is usually junk
    if len(text) > 50:
        upper_ratio = sum(1 for c in text if c.isupper()) / sum(1 for c in text if c.isalpha())
        if upper_ratio > 0.5:
            return False, f"Too many uppercase letters ({upper_ratio:.1%} > 50%)"
    
    return True, "Passed"


print(f"{'Text':<12s} {'Result':>8s} {'Reason'}")
print("-" * 55)
for sample in samples:
    passed, reason = quality_check(sample['text'])
    status = "Keep" if passed else "Discard"
    print(f"{sample['name']:<12s} {status:>8s}  {reason}")

print()
print("Real systems typically have 20-50 such rules.")
print("This can filter out about 60-80% of Common Crawl text.")

#### 2.2 Model-Based Filtering -- Hiring a "Language Teacher" to Score

Besides manual rules, you can also use a **pre-trained lightweight language model** (such as KenLM) to score each article.

The idea is the same as grading English reading comprehension: a teacher can tell at a glance whether an article is "written by a human" or "randomly generated."

```
Use a language model to compute the article's Perplexity (PPL):
  PPL very low (< 10):  Article is too simple, like "a a a a a..." -> discard
  PPL normal (10-1000): Looks like normal human writing -> keep
  PPL very high (> 1000): Gibberish -> discard
```

Some systems also train a binary classifier: Wikipedia articles = good (positive examples), random Common Crawl articles = bad (negative examples). After training, it scores each article.

**Key insight**: Wikipedia = "textbook-level" high-quality text. Use it as a yardstick to measure other text.

## 3. Deduplication

#### 3.1 Why Is Deduplication Important?

There is a lot of duplicate content on the internet:
- The same news article republished by 50 websites (Google News syndication)
- The same code snippet appearing in countless blog posts
- The same "Lorem ipsum" placeholder text
- The same Cookie notice ("This website uses cookies...")

Without deduplication:
1. The model spends precious training compute on duplicate content
2. Repeated content makes the model "memorize" rather than "understand"
3. Training data statistics become distorted (a passage's weight is a hundred times what it should be)

#### 3.2 Two Levels of Deduplication

In [ ]:
import hashlib
# === Exact deduplication ===
print("=== Level 1: Exact Dedup ===")
print()

# Simulate 5 articles, 2 of which are duplicates
docs = [
    "Machine learning is a subset of artificial intelligence.",
    "Deep learning uses neural networks with many layers.",
    "Machine learning is a subset of artificial intelligence.",  # Same as doc 1
    "Natural language processing deals with text data.",
    "Deep learning uses neural networks with many layers.",  # Same as doc 2
]

print(f"Total: {len(docs)} articles")
print()

# Hash-based deduplication
seen = set()
unique_docs = []

for i, doc in enumerate(docs):
    # SHA256 hash: turn the article into a unique fingerprint
    fingerprint = hashlib.sha256(doc.encode()).hexdigest()[:16]  # Only show first 16 chars
    is_new = fingerprint not in seen
    
    print(f"Doc {i+1}: hash={fingerprint}  {'Keep' if is_new else 'Duplicate, discard'}")
    
    if is_new:
        seen.add(fingerprint)
        unique_docs.append(doc)

print()
print(f"After dedup: {len(unique_docs)} articles ({len(docs) - len(unique_docs)} deleted)")
print()
print("Exact dedup can remove about 5-15% of Common Crawl data")
print("But that's not enough -- most duplicates are 'rewrites' rather than 'verbatim copies'")

In [ ]:
import hashlib
# === MinHash approximate deduplication: manual calculation of principles ===
print("=== Level 2: MinHash Approximate Dedup ===")
print()
print("Problem: 10 billion articles, how to quickly find similar ones?")
print("      Pairwise comparison = 10 billion^2 = impossible")
print()
print("MinHash idea: compute a 'fingerprint' for each article, similar fingerprints -> similar articles")
print()

# === Manual MinHash demonstration ===
print("=== MinHash Manual Calculation ===")
print()

# Three articles
doc_A = "the cat sat on the mat and looked at the dog"
doc_B = "the cat sat on the mat and watched the dog"  # Only one word different from A
doc_C = "quantum mechanics describes behavior of subatomic particles"  # Completely different topic

print(f"Document A: {doc_A}")
print(f"Document B: {doc_B}")
print(f"Document C: {doc_C}")
print()

# Step 1: Split each article into n-gram sets (consecutive groups of 3 words)
def get_ngrams(text, n=3):
    words = text.lower().split()
    return set(' '.join(words[i:i+n]) for i in range(len(words) - n + 1))

A_ngrams = get_ngrams(doc_A, 3)
B_ngrams = get_ngrams(doc_B, 3)
C_ngrams = get_ngrams(doc_C, 3)

print(f"Document A n-grams ({len(A_ngrams)} items): {A_ngrams}")
print()
print(f"Document B n-grams ({len(B_ngrams)} items): {B_ngrams}")
print()

# Step 2: Compute Jaccard similarity (intersection/union)
def jaccard(s1, s2):
    inter = len(s1 & s2)
    union = len(s1 | s2)
    return inter / union if union > 0 else 0

j_AB = jaccard(A_ngrams, B_ngrams)
j_AC = jaccard(A_ngrams, C_ngrams)

print(f"A intersection B size: {len(A_ngrams & B_ngrams)}")
print(f"A union B size: {len(A_ngrams | B_ngrams)}")
print(f"Jaccard(A, B) = {len(A_ngrams & B_ngrams)}/{len(A_ngrams | B_ngrams)} = {j_AB:.2%}")
print()
print(f"Jaccard(A, C) = {j_AC:.2%}")
print()

# Step 3: MinHash fingerprint (super simplified version -- using random hash simulation)
print("=== MinHash Signature Calculation ===")
print()

# Simulation: hash each n-gram to an integer, take the minimum K hash values for each article as the signature
def minhash_signature(ngrams, num_hashes=4):
    """
    Generate MinHash signature for an n-gram set
    Real MinHash uses random permutation simulation; here we use different-seed hashes to simulate
    """
    sig = []
    for i in range(num_hashes):
        # For each n-gram, hash with seed=i, take the minimum value
        min_val = float('inf')
        for ng in ngrams:
            h = hash(ng + str(i)) % 100000
            min_val = min(min_val, h)
        sig.append(min_val) if min_val != float('inf') else sig.append(0)
    return sig

sig_A = minhash_signature(A_ngrams)
sig_B = minhash_signature(B_ngrams)
sig_C = minhash_signature(C_ngrams)

print(f"A's MinHash signature: {sig_A}")
print(f"B's MinHash signature: {sig_B}")
print(f"C's MinHash signature: {sig_C}")
print()

# MinHash similarity = proportion of matching signature values
def minhash_sim(s1, s2):
    matches = sum(1 for a, b in zip(s1, s2) if a == b)
    return matches / len(s1)

mh_AB = minhash_sim(sig_A, sig_B)
mh_AC = minhash_sim(sig_A, sig_C)

print(f"MinHash approximate similarity A-B: {mh_AB:.2%}  (Exact Jaccard: {j_AB:.2%})")
print(f"MinHash approximate similarity A-C: {mh_AC:.2%}  (Exact Jaccard: {j_AC:.2%})")
print()
print("-> MinHash turns 'pairwise n-gram comparison' into 'comparing 4 numbers'")
print("-> Speed improvement of several orders of magnitude! Similarity > threshold (e.g. 80%) -> keep only one")

## 4. Data Mixing

Suppose you now have various data sources:

| Source | Volume | Quality | Purpose |
|:---|:---|:---|:---|
| Common Crawl (filtered) | 10T tokens | Medium | General knowledge, diversity |
| Wikipedia | 0.1T tokens | Very high | Factual accuracy |
| Books | 0.5T tokens | High | Long-form coherence |
| Code (GitHub) | 1T tokens | Medium | Logical reasoning |
| Academic Papers | 0.05T tokens | Very high | Scientific reasoning |

#### 4.1 How to Mix? Not Just Dumping by Raw Volume

Strategy: **High-quality data can be trained on more times (more epochs), low-quality data fewer times.**

In [ ]:
# === Data mixing strategy ===
print("=== Data Mixing: How to Configure Proportions ===")
print()

sources = [
    ("Common Crawl (filtered)", 10000, 0.6, 1),
    ("Wikipedia",               100, 0.95, 4),
    ("Books",                   500, 0.85, 2),
    ("Code (GitHub)",          1000, 0.75, 2),
    ("ArXiv Papers",             50, 0.9,  4),
    ("News",                    300, 0.7,  1),
]

print(f"{'Source':<25s} {'Raw Vol':>10s} {'Quality':>6s} {'Epoch':>6s} {'Effective':>10s} {'Share':>8s}")
print("-" * 72)

total_effective = 0
results = []
for name, size, quality, epochs in sources:
    effective = size * epochs
    total_effective += effective
    results.append((name, size, quality, epochs, effective))

for name, size, quality, epochs, effective in results:
    ratio = effective / total_effective * 100
    print(f"{name:<25s} {size:>6.0f}B   {quality:>5.0%}  {epochs:>4d}x  {effective:>8.0f}B   {ratio:>6.1f}%")

print()
print(f"Total effective data: {total_effective:.0f}B tokens")
print()
print("Key decisions:")
print("  * Wikipedia is only 100B, but epoch=4 -> actually fed 400B (high quality, train more)")
print("  * Common Crawl is 10T, but epoch=1 -> train only once (avoid garbage)")
print("  * ArXiv is small but high quality, epoch=4 -> amplify scientific reasoning ability")
print()
print("Note: More epochs does not mean copying the article 4 times")
print("      Rather, train for 4 rounds with re-shuffling -> model sees different order each time")


## 5. Complete Pipeline in Practice: Preparing Data for a 1B Model

The previous four sections covered text extraction, quality filtering, deduplication, and data mixing separately. But in real projects, these steps don't run independently -- they form a pipeline where each step's output is the next step's input. A problem in any step will affect the final training result.

Below we string the entire process together, simulating a complete data engineering pipeline: starting from raw Common Crawl WARC files, through extraction -> filtering -> dedup -> mixing, ultimately producing data that can be fed directly into training. At each step we output statistics, giving an intuitive sense of "how much was processed, how much was discarded."

Real industrial pipeline code is large (typically thousands of lines). Here we use a simplified version to demonstrate the core logic. The actual effect of each rule can be verified through the output statistics.

In [ ]:
print("=== Practice: 1B LLM Data Pipeline ===")
print()

steps = [
    ("Step 1: Determine data budget", [
        "Chinchilla optimal: N = 1B -> D = 20B tokens",
        "Over-training: N = 1B -> D = 100B tokens",
        "Choice: 50B tokens (compromise, good cost-performance)",
    ]),
    ("Step 2: Download Common Crawl", [
        "Download the latest 2-3 month dumps (~20TB compressed WARC)",
        "Tools: cc_downloader, HuggingFace datasets",
    ]),
    ("Step 3: Text extraction + language filtering", [
        "WARC -> HTML -> plain text (trafilatura / resiliparse)",
        "Language detection (fastText): keep only English and Chinese",
        "Output: ~2TB plain text (~400B tokens)",
    ]),
    ("Step 4: Quality filtering", [
        "Heuristic rules: length/word length/special chars/line repetition",
        "KenLM PPL filtering: 10 < PPL < 1000",
        "Output: ~200GB (~40B tokens) -> only 10% remains",
    ]),
    ("Step 5: Deduplication", [
        "Exact dedup: SHA256 hash -> remove ~10%",
        "MinHash approximate dedup: similarity > 80% keep only one -> remove ~20%",
        "Output: ~140GB (~28B tokens)",
    ]),
    ("Step 6: Mix with other sources", [
        "Wikipedia (2x epoch): 4B tokens",
        "Books (2x epoch): 6B tokens",
        "Code GitHub (2x epoch): 10B tokens",
        "Other: 2B tokens",
        "Total: 28B + 22B = 50B tokens",
    ]),
    ("Step 7: Tokenize + pack", [
        "Use BPE tokenizer to convert text to token IDs",
        "Concatenate into continuous sequences, cut into 2048/4096-length chunks",
        "Insert <EOS> token at document boundaries",
        "Shuffle + pack into training batches -> start training!",
    ]),
]

for title, details in steps:
    print(title)
    for d in details:
        print(f"  {d}")
    print()

print("Compression ratio summary:")
print("  20TB WARC -> 2TB plain text -> 200GB after filtering -> 140GB after dedup")
print("  Final usable data is only ~0.7% of the original download")

## 6. Data Quality > Quantity

This is a conclusion repeatedly verified in the NLP field:

```
T5 paper (2019):
  750GB of cleaned data trained better than 6TB of uncleaned data
  -> 1/8 of the data volume, but higher quality actually performed better

Textbooks Are All You Need (2023):
  A 1.3B model trained on 7B high-quality "textbook"-style data
  outperformed larger models trained on more data in coding ability
  -> Data quality can partially compensate for parameter count

LLaMA 2 -> LLaMA 3 (2023-2024):
  Model architecture barely changed; the biggest improvements were in data quality and scale
  From 2T -> 15T tokens, while data quality actually improved
  -> Better filtering, better deduplication, better mixing strategies
```

The practical implication is straightforward: if you have limited resources, prioritize investing in data quality improvement rather than blindly expanding data volume. The training value of 100 good books far exceeds that of 10,000 spam emails.

## 7. From Tokenize to Training Stream

The final step of data engineering is tokenize. What does the data look like after this?

```
Document A: "Machine learning is great."
  -> tokenize -> [42, 567, 18, 891, 15]

Document B: "Deep learning is also great."
  -> tokenize -> [123, 567, 18, 456, 891, 15]

Then concatenate into a "token noodle":
[42, 567, 18, 891, 15, <EOS>, 123, 567, 18, 456, 891, 15, <EOS>, ...]

Cut into training chunks (assume seq_len=8):
  Chunk 1: [42, 567, 18, 891, 15, <EOS>, 123, 567]
  Chunk 2: [18, 456, 891, 15, <EOS>, ...]
  ...

Note: chunk boundaries can cut through documents!
  The last 3 tokens of Chunk 1 come from Document B, the first 5 from Document A
  -> The model sometimes "cross-document" learns within a chunk
  -> Solution: add <EOS> to tell the model "new document starts"
```

#### 7.1 Sequence Packing: Packing Short Documents into Full Chunks

The previous section showed the naive approach of "concatenate -> cut into fixed lengths." But it has a problem: if document lengths vary widely, some chunks will be filled with lots of padding, wasting compute.

**Sequence Packing** solves this problem with a bin-packing algorithm: tightly pack multiple short documents into a fixed-length chunk, leaving as little empty space as possible.

```
Naive approach (with padding waste):
  Chunk 1: [Doc A (500 tokens)][padding (3596 tokens)]  <- 88% wasted
  Chunk 2: [Doc B (800 tokens)][padding (3296 tokens)]  <- 80% wasted

Packing approach (fill as much as possible):
  Chunk 1: [Doc A (500)][Doc B (800)][Doc C (1200)][Doc D (1500)][padding (96)]  <- 98% utilization
```

Practical benefit: training throughput can improve by **2x~4x** (more benefit with more short documents).

But Packing has a technical problem that must be solved: **different documents cannot attend to each other, otherwise there is information leakage.** The solution is to add a block-diagonal mask during attention computation.

In [ ]:
# === Sequence Packing manual calculation demonstration ===
print("=== Sequence Packing: Bin-Packing Algorithm Manual Calculation ===")
print()

# Simulate token lengths of 8 documents
docs = [
    ("Doc A", 120),
    ("Doc B", 80),
    ("Doc C", 200),
    ("Doc D", 50),
    ("Doc E", 150),
    ("Doc F", 90),
    ("Doc G", 60),
    ("Doc H", 40),
]

max_seq_len = 256  # Fixed chunk length

# First-Fit Decreasing bin packing: sort long docs first, then try to fit into existing chunks
sorted_docs = sorted(docs, key=lambda x: -x[1])  # Sort by length descending
chunks = []  # Each chunk is (doc list, used length)

for name, length in sorted_docs:
    placed = False
    # Try to fit into an existing chunk
    for chunk in chunks:
        if chunk[1] + length + 1 <= max_seq_len:  # +1 for EOS token
            chunk[0].append((name, length))
            chunk[1] += length + 1
            placed = True
            break
    if not placed:
        chunks.append([[(name, length)], length + 1])

print(f"Total documents: {len(docs)}")
print(f"Chunk capacity: {max_seq_len} tokens")
print(f"Packing result: {len(chunks)} chunks")
print()

total_used = 0
for i, (items, used) in enumerate(chunks):
    doc_names = ' + '.join(name for name, _ in items)
    padding = max_seq_len - used
    util = used / max_seq_len * 100
    total_used += used
    print(f"  Chunk {i+1}: [{doc_names}]")
    print(f"          Used {used} tokens, padding {padding}, utilization {util:.0f}%")

overall_util = total_used / (len(chunks) * max_seq_len) * 100
print(f"\nOverall utilization: {overall_util:.0f}%")
print()
print("Compare with naive approach (one doc per chunk):")
naive_chunks = len(docs)
print(f"  Naive: {naive_chunks} chunks")
print(f"  Packing: {len(chunks)} chunks")
print(f"  Saved: {(1 - len(chunks)/naive_chunks)*100:.0f}%")

#### 7.2 Block-Diagonal Attention Mask: Preventing Documents from Seeing Each Other

After packing, a single chunk may contain 3-4 different documents. Without any restrictions, the model's attention can "see" content from other documents -- this is information leakage.

The solution: **block-diagonal mask**. Each document can only attend within itself; attention weights between different documents are set to negative infinity (becoming 0 after softmax).

```
Chunk: [Doc A][Doc B][Doc C]

Attention Mask:
        A A A B B C C
    A [ 1 1 1 0 0 0 0 ]   <- A only sees A
    A [ 1 1 1 0 0 0 0 ]
    A [ 1 1 1 0 0 0 0 ]
    B [ 0 0 0 1 1 0 0 ]   <- B only sees B
    B [ 0 0 0 1 1 0 0 ]
    C [ 0 0 0 0 0 1 1 ]   <- C only sees C
    C [ 0 0 0 0 0 1 1 ]

Blocks on the diagonal are each independent = block-diagonal
```

In [ ]:
import numpy as np

# === Block-Diagonal Mask construction demonstration ===
print("=== Block-Diagonal Attention Mask ===")
print()

# Assume a packed chunk contains 3 documents with lengths 3, 2, 2 respectively
doc_lengths = [3, 2, 2]
total_len = sum(doc_lengths)  # 7

# Construct block-diagonal mask (pure numpy)
mask = np.zeros((total_len, total_len))

pos = 0
for length in doc_lengths:
    for i in range(length):
        for j in range(i + 1):
            mask[pos + i, pos + j] = 1.0
    pos += length

print(f"Document lengths: {doc_lengths}, total length: {total_len}")
print()
print("Block-Diagonal Mask (1=visible, 0=masked):")
print("      ", "  ".join([f"t{i}" for i in range(total_len)]))
labels = []
pos = 0
for doc_idx, length in enumerate(doc_lengths):
    for _ in range(length):
        labels.append(f"D{doc_idx}")
print()
for i in range(total_len):
    row = "  ".join([f" {int(mask[i,j])}" for j in range(total_len)])
    print(f"  {labels[i]} t{i}: {row}")

print()
print("Observations:")
print("  D0 (Doc 0) internal: t0->t0, t1->t0,t1, t2->t0,t1,t2  <- causal attention")
print("  D1 (Doc 1) internal: t3->t3, t4->t3,t4              <- completely isolated from D0")
print("  D2 (Doc 2) internal: t5->t5, t6->t5,t6              <- isolated from other docs")
print()
print("In actual training, you also need:")
print("  1. Position IDs reset to 0 for each document")
print("  2. Loss computed only on real tokens, not on padding")

**Fill-in-the-Middle (FIM) -- Data Format for Training Code Completion Models**

Everything discussed so far concerns the data flow of a general language model: left-to-right, autoregressive next-token prediction. But for code models, there is another important data construction method.

When writing code, there is code both before and after the cursor. On the left is the already-written part (prefix), and on the right is the unchanged part (suffix). A code completion model needs to understand both sides to accurately fill in the middle. Fill-in-the-Middle (FIM) is the training data format designed for this scenario.

The approach is to randomly split each code segment into three parts and rearrange them with special tokens:

```
<PRE> prefix content <SUF> suffix content <MID> middle content
```

During training, after the model sees `<PRE> ... <SUF> ... <MID>`, it must predict the removed middle portion. The labels for the prefix and suffix portions are set to ignore_index -- the model can see this context but no loss is computed for them. Only the middle portion has loss computed normally.

For example, a 5-line code segment:

```
Original code:
  a = 1
  b = 2
  c = 3
  d = 4
  e = 5

Split into first 2 + middle 2 + last 1:
  prefix = "a = 1\nb = 2\n"     (model sees, does not predict)
  middle = "c = 3\nd = 4\n"     (training target)
  suffix = "e = 5"              (model sees, does not predict)

FIM format (model input):
  <PRE> a = 1\nb = 2\n <SUF> e = 5 <MID> c = 3\nd = 4\n
  |- can see, not predicting -|  |- can see -|  |--- to predict ---|
```

Not all samples use FIM. Typically about 50% use standard autoregressive format (maintaining general language understanding and generation ability), and 50% use FIM format (learning to do code completion by looking at both prefix and suffix). The two formats are mixed during training, so the model can both hold normal conversations and do code completion in an IDE. FIM does not affect the tokenize and packing pipeline -- it is a text-level rearrangement done before tokenization.

Code models (Code Llama, DeepSeek-Coder, StarCoder, etc.) all use FIM as a standard configuration. General text models typically do not, because writing scenarios are naturally left-to-right.

In [ ]:
import numpy as np

# FIM format conversion manual calculation demonstration

def fim_transform(code):
    """
    Convert a piece of code to FIM training format
    
    Randomly select split points -> split into prefix/middle/suffix -> rearrange in FIM format
    """
    lines = code.split('\n')
    n = len(lines)
    
    # Randomly select split points
    prefix_len = np.random.randint(1, max(2, int(n * 0.4)))
    middle_len = np.random.randint(1, max(2, int(n * 0.5)))
    prefix_len = min(prefix_len, n - 1)
    middle_len = min(middle_len, n - prefix_len)
    
    prefix = '\n'.join(lines[:prefix_len])
    middle = '\n'.join(lines[prefix_len:prefix_len + middle_len])
    suffix = '\n'.join(lines[prefix_len + middle_len:])
    
    fim_text = f"<PRE> {prefix}\n<SUF> {suffix}\n<MID> {middle}"
    return fim_text, prefix, middle, suffix


np.random.seed(42)

sample_code = """def calculate(x, y):
    z = x + y
    result = z * 2
    if result > 100:
        return result
    else:
        return 0

print(calculate(3, 4))"""

fim_text, prefix, middle, suffix = fim_transform(sample_code)

print("=== FIM Format Conversion Manual Calculation ===")
print()
print("Original code:")
print(sample_code)
print()
print("--- After splitting ---")
print(f"Prefix ({len(prefix)} chars, model sees):")
print(prefix)
print()
print(f"Middle ({len(middle)} chars, training target):")
print(middle)
print()
print(f"Suffix ({len(suffix)} chars, model sees):")
print(suffix if suffix else '(empty)')
print()
print("--- FIM format (model input) ---")
print(fim_text)
print()

print("=== Label Mask During Training ===")
print()
print("Rules:")
print("  <PRE> ... <SUF> ... <MID> these special tokens -> model knows but doesn't predict")
print("  prefix and suffix content     -> label = ignore_index")
print("  middle content                -> label = normal token (participates in loss)")
print()
print("Key observations:")
print("1. FIM rearranges text so the model can leverage bidirectional context during training")
print("2. <PRE> and <SUF> provide prefix and suffix info, after <MID> the model starts predicting the removed part")
print("3. Only the middle region's loss is kept -- prefix/suffix give info but no penalty")
print("4. About 50% samples use standard format + 50% use FIM -> balance autoregressive generation and code completion")

## Summary

Confirm you understand these points (check in order):

1. [ ] Data sources: Common Crawl (main) + Wikipedia + Books + Code + Papers
2. [ ] Pipeline five steps: text extraction -> language filtering -> quality filtering -> deduplication -> data mixing
3. [ ] HTML -> Text: remove script/style tags + remove HTML tags + clean whitespace
4. [ ] Quality filtering: heuristic rules (length/word length/symbols) + PPL (language model scoring)
5. [ ] Exact deduplication: SHA256 hashing, keep only one of identical documents
6. [ ] MinHash: turn articles into fingerprints (n-gram -> hash -> take minimum K), similar fingerprints = similar articles
7. [ ] Data mixing: high quality gets more epochs, low quality gets fewer epochs
8. [ ] After tokenize: concatenate into "token noodle" -> cut into fixed-length chunks -> start training
9. [ ] FIM (Fill-in-the-Middle): rearrange text with <PRE>/<SUF>/<MID>, training code completion ability

**One-sentence summary**: Data engineering is the least glamorous but most critical part of LLM training. Architectures can be copied, but the data pipeline is each company's core competitive advantage. 20TB of raw data ends up with only 0.7% usable -- washing the ingredients takes more effort than cooking.

## Exercises

To be added.